# Quan hệ giữa thông tin trong web traffic và rev

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

# ==========================================
# 1. ĐỌC VÀ XỬ LÝ DỮ LIỆU
# ==========================================
# Đọc 2 file dữ liệu
df_sales = pd.read_csv('../dataset/sales.csv')
df_web = pd.read_csv('../dataset/web_traffic.csv')

# Xử lý ngày tháng cho Sales và tổng hợp Doanh thu theo tháng
df_sales['Date'] = pd.to_datetime(df_sales['Date'])
df_sales['year_month'] = df_sales['Date'].dt.to_period('M')
monthly_sales = df_sales.groupby('year_month')['Revenue'].sum().reset_index()

# Xử lý ngày tháng cho Web Traffic và tổng hợp các chỉ số theo tháng
df_web['date'] = pd.to_datetime(df_web['date'])
df_web['year_month'] = df_web['date'].dt.to_period('M')
monthly_web = df_web.groupby('year_month').agg(
    total_sessions=('sessions', 'sum'),
    total_unique_visitors=('unique_visitors', 'sum'),
    total_page_views=('page_views', 'sum'),
    avg_bounce_rate=('bounce_rate', 'mean')
).reset_index()

# ==========================================
# 2. GỘP DỮ LIỆU VÀ CHUẨN BỊ VẼ
# ==========================================
# Gộp 2 bảng lại theo tháng
df_marketing_sales = pd.merge(monthly_sales, monthly_web, on='year_month', how='inner')

# Chuyển Period sang Datetime để vẽ trục X không bị lỗi
df_marketing_sales['month_dt'] = df_marketing_sales['year_month'].dt.to_timestamp()

# Tính giá trị mỗi phiên (Revenue Per Session)
df_marketing_sales['Revenue_Per_Session'] = df_marketing_sales['Revenue'] / df_marketing_sales['total_sessions']

# ==========================================
# 3. TRỰC QUAN HÓA (VISUALIZATION)
# ==========================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('MỐI QUAN HỆ GIỮA LƯU LƯỢNG TRUY CẬP WEB VÀ DOANH THU', fontsize=18, fontweight='bold')

# --- BIỂU ĐỒ 1: Xu hướng Sessions và Revenue theo thời gian (Dual-axis) ---
ax1 = axes[0, 0]
ax1.plot(df_marketing_sales['month_dt'], df_marketing_sales['total_sessions'], color='teal', marker='o', linewidth=2, label='Sessions')
ax1.set_xlabel('Tháng')
ax1.set_ylabel('Lượt truy cập (Sessions)', color='teal')
ax1.tick_params(axis='y', labelcolor='teal')

ax2 = ax1.twinx()
ax2.plot(df_marketing_sales['month_dt'], df_marketing_sales['Revenue'], color='blue', marker='s', linestyle='--', linewidth=2, label='Revenue')
ax2.set_ylabel('Doanh thu ($)', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
ax1.set_title('Xu hướng Lượt truy cập và Doanh thu theo tháng', fontsize=14)

# --- BIỂU ĐỒ 2: Scatter Plot (Lượt truy cập vs Doanh thu) ---
sns.regplot(
    data=df_marketing_sales, x='total_sessions', y='Revenue', 
    ax=axes[0, 1], scatter_kws={'s': 100, 'alpha': 0.7, 'color': 'purple'}, line_kws={'color': 'red'}
)
axes[0, 1].set_title('Mức độ tương quan: Lượt truy cập tỷ lệ với Doanh thu?', fontsize=14)
axes[0, 1].set_xlabel('Tổng Lượt truy cập (Sessions)')
axes[0, 1].set_ylabel('Doanh thu ($)')

# --- BIỂU ĐỒ 3: Scatter Plot (Tỷ lệ thoát trang vs Giá trị mỗi phiên) ---
sns.scatterplot(
    data=df_marketing_sales, x='avg_bounce_rate', y='Revenue_Per_Session', 
    size='total_sessions', sizes=(50, 400), alpha=0.7, ax=axes[1, 0], color='orange'
)
# Thêm đường xu hướng
sns.regplot(data=df_marketing_sales, x='avg_bounce_rate', y='Revenue_Per_Session', scatter=False, ax=axes[1, 0], color='red')
axes[1, 0].set_title('Tác động của Tỷ lệ thoát trang đến Giá trị mỗi phiên truy cập', fontsize=14)
axes[1, 0].set_xlabel('Tỷ lệ thoát trang trung bình (Bounce Rate)')
axes[1, 0].set_ylabel('Doanh thu trên mỗi lượt truy cập ($)')

# --- BIỂU ĐỒ 4: Ma trận tương quan (Heatmap) ---
corr_cols = [
    'Revenue', 'total_sessions', 'total_unique_visitors', 
    'total_page_views', 'avg_bounce_rate', 'Revenue_Per_Session'
]
corr_matrix = df_marketing_sales[corr_cols].corr()

sns.heatmap(
    corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", 
    vmin=-1, vmax=1, linewidths=.5, ax=axes[1, 1]
)
axes[1, 1].set_title('Bản đồ nhiệt: Tương quan giữa Web Traffic và Sales', fontsize=14)

# Căn chỉnh hiển thị
plt.tight_layout()
fig.subplots_adjust(top=0.92)
plt.show()

# Tương quan giữa truy cập vào web và doanh thu
web_traffic.csv + sales (correlation between sales and methods)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

# ==========================================
# 1. XỬ LÝ DỮ LIỆU SALES (NHƯ CŨ)
# ==========================================
df_sales = pd.read_csv('../dataset/sales.csv')
df_sales['Date'] = pd.to_datetime(df_sales['Date'])
df_sales['year_month'] = df_sales['Date'].dt.to_period('M')
monthly_sales = df_sales.groupby('year_month')['Revenue'].sum().reset_index()

# ==========================================
# 2. XỬ LÝ WEB TRAFFIC THEO TỪNG PHƯƠNG THỨC (TRAFFIC SOURCE)
# ==========================================
df_web = pd.read_csv('../dataset/web_traffic.csv')
df_web['date'] = pd.to_datetime(df_web['date'])
df_web['year_month'] = df_web['date'].dt.to_period('M')

# Xoay bảng (Pivot) để mỗi nguồn truy cập là một cột, giá trị là tổng sessions
traffic_pivot = df_web.pivot_table(
    index='year_month', 
    columns='traffic_source', 
    values='sessions', 
    aggfunc='sum'
).fillna(0).reset_index()

# Đổi tên các cột để dễ nhận biết (VD: sessions_organic_search)
traffic_pivot.columns = [f"sessions_{col}" if col != 'year_month' else col for col in traffic_pivot.columns]

# ==========================================
# 3. GỘP DỮ LIỆU & TRỰC QUAN HÓA
# ==========================================
df_final = pd.merge(monthly_sales, traffic_pivot, on='year_month', how='inner')
df_final['month_dt'] = df_final['year_month'].dt.to_timestamp()

sns.set_theme(style="white")
fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(2, 2)

# --- BIỂU ĐỒ 1: Phân bổ nguồn truy cập theo thời gian (Stacked Area Chart) ---
ax1 = fig.add_subplot(gs[0, 0])
session_cols = [c for c in df_final.columns if 'sessions_' in c]
ax1.stackplot(df_final['month_dt'], [df_final[c] for c in session_cols], labels=[c.replace('sessions_', '') for c in session_cols], alpha=0.8)
ax1.set_title('Cấu trúc nguồn truy cập qua các tháng', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# --- BIỂU ĐỒ 2: Doanh thu vs Tổng lượt truy cập ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(df_final['month_dt'], df_final['Revenue'], color='blue', marker='o', linewidth=3, label='Revenue')
ax2_twin = ax2.twinx()
total_sessions = df_final[session_cols].sum(axis=1)
ax2_twin.bar(df_final['month_dt'], total_sessions, color='gray', alpha=0.3, width=15, label='Total Sessions')
ax2.set_title('Tương quan Doanh thu và Tổng lượt truy cập', fontsize=14, fontweight='bold')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# --- BIỂU ĐỒ 3: Ma trận tương quan chi tiết giữa từng nguồn và Doanh thu ---
ax3 = fig.add_subplot(gs[1, :])
# Chỉ lấy các cột sessions và Revenue để tính tương quan
corr_cols = ['Revenue'] + session_cols
corr_matrix = df_final[corr_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='YlGnBu', fmt=".2f", ax=ax3)
ax3.set_title('Mức độ ảnh hưởng của từng nguồn truy cập tới Doanh thu', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()